In [1]:
import app
import os
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import box
import rasterio
import rasterio.plot as rplt
import rasterio.windows  as rw
from rasterio.transform import Affine
from matplotlib_scalebar.scalebar import ScaleBar
import numpy as np
import cmocean as cmo

app.setup_logger(use_console_handler=False, use_file_handler=True)

import logging
logger = logging.getLogger(__name__)
logger.setLevel("INFO")

In [2]:
base_directory = r"D:\PhD\21_Experiments\TidesDamageDriver"
drainages_folder = os.path.join(
    base_directory, "03_cleaned", "drainages"
)
drainages_ref_folder = os.path.join(
    base_directory, "02_processed", "02_drainages"
)
data_img_folder = os.path.join(
    base_directory, "01_raw", "L8S2S1-events"
)
data_plot_folder = os.path.join(
    base_directory, "03_cleaned", "drainages_plots"
)

In [3]:
gdf = app.lakes.plotting.get_geodataframes(drainages_folder)[0]
gdf_ref = app.lakes.plotting.get_geodataframes(drainages_ref_folder, ["_e."])[0]
gdf

,label,index,year,lon,lat,start_date,end_date,mean-0,median-0,area-0,volume-0,mean-1,median-1,volume-1,dmg,act,lai,fuerst,fuerst+lai,geometry
0,A,29,2019,95.688019,-66.671201,2019-12-24,2019-12-31,1.130955,1.104492,66600.0,75321.609664,0.000000,0.000000,0.000000,0.235667,0.269375,NaN,NaN,NaN,"POLYGON ((95.6809 -66.67233, 95.68083 -66.6720..."
1,B,28,2019,95.737264,-66.660983,2019-12-22,2019-12-31,1.283588,1.089844,55800.0,71624.206638,0.313599,0.335693,1411.193848,0.333090,0.458099,1.0,0.0,1.0,"POLYGON ((95.73322 -66.6612, 95.73302 -66.6604..."
2,C,39,2019,95.759349,-66.646830,2020-01-07,2020-01-13,0.569657,NaN,NaN,64086.362457,0.307686,NaN,6092.188454,0.333090,0.458099,1.0,0.0,1.0,"POLYGON ((95.75933 -66.64674, 95.75935 -66.646..."
3,D,8,2016,96.040652,-66.571884,2017-01-27,2017-02-03,1.441006,1.357422,90900.0,130987.440419,0.222290,0.222290,600.183105,0.356259,0.574090,1.0,0.0,1.0,"POLYGON ((96.03223 -66.57217, 96.03216 -66.571..."
4,E,32,2019,97.782530,-66.594982,2020-01-30,2020-02-03,2.299795,2.486328,154800.0,356008.215523,0.631179,0.626221,14769.579649,0.139619,0.533078,1.0,0.0,1.0,"POLYGON ((97.7797 -66.5975, 97.77961 -66.59723..."
5,F,40,2019,98.104002,-66.492315,2020-01-30,2020-02-03,1.125474,NaN,NaN,109621.166992,0.495280,NaN,5646.196747,0.169961,0.513592,1.0,1.0,2.0,"POLYGON ((98.10397 -66.49223, 98.10399 -66.492..."
6,G,41,2019,98.493578,-65.782618,2020-01-04,2020-01-11,0.583205,NaN,NaN,30501.635742,NaN,NaN,0.000000,0.430535,0.801361,1.0,0.0,1.0,"POLYGON ((98.49355 -65.78253, 98.49357 -65.782..."
7,H,17,2018,98.847890,-66.359584,2019-01-31,2019-02-08,0.914807,0.891113,99900.0,91389.173824,0.595006,0.585693,7497.070205,0.284860,0.313371,0.0,0.0,0.0,"POLYGON ((98.84161 -66.36032, 98.84131 -66.359..."
8,I,26,2019,98.870325,-66.357777,2020-01-20,2020-01-27,0.860971,0.859863,93600.0,80586.860847,0.818970,0.876709,7370.727539,0.256579,0.386947,0.0,0.0,0.0,"POLYGON ((98.86448 -66.35728, 98.86428 -66.356..."
9,J,23,2019,99.719303,-66.264245,2020-01-04,2020-01-11,0.992884,1.030273,59400.0,58977.294266,0.483887,0.489014,2612.988281,0.135486,0.681241,0.0,0.0,0.0,"POLYGON ((99.71542 -66.26532, 99.71531 -66.265..."


In [6]:
for row in gdf.iterrows():
    if row[0] < 0:
        continue
    if row[0] == 50:
        break
    
    
    centroid = row[1].geometry.centroid
    event_folder = os.path.join(data_img_folder, str(row[1]["index"]))
    # count_dpg = len([f for f in os.listdir(event_folder) if "T47DPG.tif" in f])
    # count_dng = len([f for f in os.listdir(event_folder) if "T47DNG.tif" in f])

    # if count_dpg > count_dpg:
    #     filenames = [f for f in os.listdir(event_folder) if "T47DNG.tif" not in f]
    # else:
    #     filenames = [f for f in os.listdir(event_folder) if "T47DPG.tif" not in f]
    filenames = [f for f in os.listdir(event_folder)]
    filenames.sort(key=lambda x: app.lakes.tiffiles.parse_filename(x)[1])  # Sort by date

    row_ref = gdf_ref.iloc[row[1]["index"]]
    ref_date = row_ref["date-0"]
    nerby_optical, nerby_s1 = app.lakes.tiffiles.get_nearby_images(filenames, ref_date, (11, 14), (4, 6))
    try:
        lake_id = "-".join([str(row_ref['ifile_0']), str(row_ref['ifile_1']), str(int(row_ref['lake id']))])
    except Exception as e:
        logger.warning(f"Could not parse lake ID for row {row[1]['index']}: {e}")
        lake_id = ""
    try:
        fig, axes = plt.subplots(5, 5, figsize=(22, 15), sharex=True, sharey=True)
        for ifname, fname in enumerate(nerby_optical):
            logger.info(f"Processing optical image {fname}")
            sat, dt = app.lakes.tiffiles.parse_filename(fname)
            ax = axes.ravel()[ifname]
            gpd.GeoDataFrame(geometry=[centroid]).plot(ax=ax, color='red', markersize=5)
            with rasterio.open(os.path.join(event_folder, fname)) as src:

                if sat == 'L8':
                    raster = src.read()
                    transform = src.transform
                elif sat == 'S2':
                    raster, meta = app.lakes.tiffiles.reproject_in_memory(src, "EPSG:3031")
                    transform = meta['transform']
                raster = raster[:3][::-1, :, :]

                rplt.show(raster, ax=ax, transform=transform)
                ax.text(0, 0.95, sat + ' | ' + dt.strftime("%Y-%m-%d %H:%M:%S"), transform=ax.transAxes,
                        fontsize=12, bbox=dict(facecolor='white'))
        
        for ax in axes.ravel():
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_xlabel("")
            ax.set_ylabel("")
            ax.set_title("")
        
        ax = axes.ravel()[0]
        ax.text(0, 0.81, row[1]["label"] + " | " + f"{row[1]['index']:02d}", transform=ax.transAxes, bbox=dict(facecolor='salmon'),
                        fontsize=12)
        
        #ax = axes.ravel()[10]
        #ax.text(0, 0.85, ref_date, transform=ax.transAxes, bbox=dict(facecolor='salmon'),
        #                fontsize=12)
        #plt.tight_layout()
        try:
            fig.savefig(os.path.join(data_plot_folder, f"{row[1]['label']}_{int(row[1]['index']):02d}_{lake_id}_optical.png"), dpi=300)
        except Exception as e:
            logger.warning(f"Could not save optical figure for row {row[1]['index']}: {e}")
            fig.savefig(os.path.join(data_plot_folder, f"{row[1]['label']}_{int(row[1]['index']):02d}_optical.png"), dpi=300)
        plt.close()
    except Exception as e:
        logger.error(row_ref)
        logger.error(row)
        logger.error(f"Error processing event {row[1]['label']}: {e}")
    
    try:
        fig, axes = plt.subplots(2, 5, figsize=(20, 8), sharex=True, sharey=True)
        for ifname, fname in enumerate(nerby_s1):
            sat, dt = app.lakes.tiffiles.parse_filename(fname)
            ax = axes.ravel()[ifname]
            gpd.GeoDataFrame(geometry=[centroid]).plot(ax=ax, color='red', markersize=5)
            with rasterio.open(os.path.join(event_folder, fname)) as src:
                # print(f"Processing {fname}...")
                s1_raster, s1_meta = app.lakes.tiffiles.reproject_in_memory(src, "EPSG:3031")
                s1_transform = s1_meta['transform']
                # raster = src.read()
                # print(src.transform)
                rplt.show(s1_raster, ax=ax, transform=s1_transform, cmap='cmo.ice')
                ax.text(0, 0.95, sat + ' | ' + dt.strftime("%Y-%m-%d %H:%M:%S"), transform=ax.transAxes,
                        fontsize=12, bbox=dict(facecolor='white'))
            # Remove axis labels and ticks
        
        for ax in axes.ravel():
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_xlabel("")
            ax.set_ylabel("")
            ax.set_title("")

        ax = axes.ravel()[0]
        ax.text(0, 0.88,row[1]["label"] + " | " + f"{row[1]['index']:02d}", transform=ax.transAxes, bbox=dict(facecolor='salmon'),
                        fontsize=12)

        plt.tight_layout()
        try:

            fig.savefig(os.path.join(data_plot_folder, f"{row[1]['label']}_{int(row[1]['index']):02d}_{lake_id}_S1.png"), dpi=300)
        except Exception as e:
            fig.savefig(os.path.join(data_plot_folder, f"{row[1]['label']}_{int(row[1]['index']):02d}_S1.png"), dpi=300)
        plt.close()
    except Exception as e:
        logger.error(row_ref)
        logger.error(row)
        logger.error(f"Error processing event {row[1]['label']}: {e}")
    